# Minimal iQPE for Ising model on the AC1000 emulator

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need the `jupyter`, `azure-quantum`.

```bash
pip install 'qdk-chemistry[jupyter,azure-quantum]'
```

In [ ]:
import numpy as np
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

from qdk_chemistry.utils import Logger

# Silence library logs
Logger.set_global_level(Logger.LogLevel.off)

### Set up the Ising model

The cell below builds a transverse-field Ising Hamiltonian on the lattice we defined: a **2-site open chain** (`num_sites = 2`, `periodic=False`), i.e. two spin-1/2 sites connected by a single bond. With $J = -1$ and $h = 1$ the Hamiltonian is

$$
H \;=\; J\,\sigma^z_0 \sigma^z_1 \;+\; h\,(\sigma^x_0 + \sigma^x_1)
\;=\; -\,\sigma^z_0 \sigma^z_1 \;+\; \sigma^x_0 + \sigma^x_1 .
$$

- The $\sigma^z\sigma^z$ term with $J = -1$ is a *ferromagnetic* coupling on the single bond — it lowers the energy when the two spins are aligned along $z$.
- The $\sigma^x$ terms with $h = 1$ are a transverse field on each site; they do not commute with the coupling, so they drive quantum fluctuations between the $z$-basis states.

Each site maps to one qubit, giving a 2-qubit Hamiltonian whose ground-state energy we estimate below with iterative quantum phase estimation (iQPE).

We deliberately pick the evolution time $t = \pi/2$ so that every rotation angle becomes a multiple of $\pi/2$, turning this into a toy model whose iQPE circuit is entirely Clifford and thus runs natively on the AC1000 Clifford-rounding emulator.

In [ ]:
# Create a ising model
num_sites = 2
J = -1.0
H = 1.0

lattice = LatticeGraph.chain(num_sites, periodic=False)
qubit_hamiltonian = create_ising_hamiltonian(lattice, j=J, h=H)

# Print the Hamiltonian as an explicit Pauli-string expansion. Each string acts
# on the `num_sites` qubits (little-endian: the rightmost character is qubit 0).
# We expect one ZZ coupling term (J) and one X term per site (h).
print(f"Qubit Hamiltonian: {qubit_hamiltonian.get_summary()}")

### Prepare the trial state

We choose the Bell-like state $|\psi\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$, which is an exact eigenstate of the Hamiltonian $H = -\,ZZ + XI + IX$ so that

$$
H\,|\psi\rangle = -1\cdot|\psi\rangle .
$$

The target eigenvalut is -1.

This state is also trivial to prepare: it is a sparse 2-term superposition, which we construct directly from its computational-basis bitstrings and amplitudes with `Wavefunction.from_bitstrings`, then compile into a state-preparation circuit with the `sparse_isometry_gf2x` algorithm.

In [ ]:
from qdk_chemistry.data import Wavefunction

# Bell-like eigenstate for E = -1:  (|00> - |11>)/sqrt(2)
wfn = Wavefunction.from_bitstrings(["00", "11"], [1/np.sqrt(2), -1/np.sqrt(2)])

state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(wfn)

### Set up Azure Quantum Workspace

We connect to the Azure Quantum workspace that hosts the AC1000 emulator. After authenticating (here with the Azure CLI credential), we fetch the emulator target and define the emulation settings: Clifford-rounding simulation with noise and timing disabled, and a fixed seed for reproducibility.

In [ ]:
from azure.identity import InteractiveBrowserCredential, AzureCliCredential
from azure.quantum import Workspace

# Azure Quantum configuration
WORKSPACE_SUBSCRIPTION_ID = "INPUT-SUBSCRIPTION-ID"
WORKSPACE_RESOURCE_GROUP = "INPUT-RESOURCE-GROUP"
WORKSPACE_NAME = "INPUT-WORKSPACE-NAME"
WORKSPACE_LOCATION = "INPUT-WORKSPACE-LOCATION"
TARGET_NAME = "INPUT-TARGET-NAME"

# credential = InteractiveBrowserCredential()
credential = AzureCliCredential()

azure_workspace = Workspace(
        subscription_id=WORKSPACE_SUBSCRIPTION_ID,
        resource_group=WORKSPACE_RESOURCE_GROUP,
        name=WORKSPACE_NAME,
        location=WORKSPACE_LOCATION,
        credential=credential,
    )
target = azure_workspace.get_targets(TARGET_NAME)

emulation_settings = {
    "simulationType": "cliffordrounding",
    "enableNoise": False,
    "emulateTiming": False,
    "seed": 42,
}

### Set up the qdk-chemistry circuit executor to use Azure Quantum Emulator backend

We wrap the emulator target in a qdk-chemistry `circuit_executor` of type `azure_quantum_emulator`, passing the emulation settings defined above. This executor is the backend the iQPE algorithm will use to compile each circuit to QIR, submit it to the AC1000 emulator, and collect the measurement counts.

In [ ]:
import json

ac1000_emulator = create(
    "circuit_executor",
    "azure_quantum_emulator",
    job_name="qdk_chemistry_iqpe_test_job",
    emulation_settings=json.dumps(emulation_settings),
)
ac1000_emulator.set_algorithm_instance("target", target)

### Set up iQPE

We assemble the iterative QPE algorithm from its building blocks: a Trotterized time-evolution unitary at $t = \pi/2$, a Pauli-sequence controlled-circuit mapper, and a `qdk_iterative` circuit builder with `num_bits = 2` bits of precision. We attach the AC1000 emulator as the circuit executor, run iQPE against the prepared state and Hamiltonian, and read off the estimated ground-state energy.

In [ ]:
M_PRECISION=2
evolution_time = np.pi / 2

unitary_builder = AlgorithmRef(
    "hamiltonian_unitary_builder", "trotter", time=evolution_time
)
circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")

circuit_builder = AlgorithmRef(
    "qpe_circuit_builder",
    "qdk_iterative",
    num_bits=M_PRECISION,
    unitary_builder=unitary_builder,
    controlled_circuit_mapper=circuit_mapper,
)

iqpe = create(
    "phase_estimation",
    "qdk_iterative",
    qpe_circuit_builder=circuit_builder
)
iqpe.set_algorithm_instance("circuit_executor", ac1000_emulator)

print(f"Running IQPE on Azure Quantum target {target.name}...")
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
)

# Summarize the QPE results
estimated_energy = result.raw_energy
print(f"\nEnergy result: {result.raw_energy}")

### Compare with qdk full state simulator

As a reference check, we run the same iQPE setup on the local `qdk_full_state_simulator`. The estimated energy should match the emulator result, confirming the Clifford circuit behaves as expected.

In [ ]:
qdk_full_state_simulator = create("circuit_executor","qdk_full_state_simulator")
iqpe_w_qdk_simulator = create(
    "phase_estimation",
    "qdk_iterative",
    qpe_circuit_builder=circuit_builder
)
iqpe_w_qdk_simulator.set_algorithm_instance("circuit_executor", qdk_full_state_simulator)
print("Running iQPE with QDK full state simulator...")
result = iqpe_w_qdk_simulator.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
)

# Summarize the QPE results
estimated_energy = result.raw_energy
print(f"Energy result: {result.raw_energy}")


### Combine all iterations into a single circuit

Instead of running one circuit per iQPE iteration, we rebuild the circuit builder with `combine_iterations=True` so that all iterations, including the mid-circuit measurements and classical feedback, are merged into one QIR program. We then render that combined circuit for inspection.

In [ ]:
from qdk.widgets import Circuit
from qdk import qsharp
from qdk._context import CircuitGenerationMethod

single_qir_circuit_builder = create(
    "qpe_circuit_builder",
    "qdk_iterative",
    num_bits=M_PRECISION,
    unitary_builder=unitary_builder,
    controlled_circuit_mapper=circuit_mapper,
    combine_iterations=True
)
circuits = single_qir_circuit_builder.run(state_preparation=state_prep_circuit, qubit_hamiltonian=qubit_hamiltonian)
full_circuit = circuits[0]

Circuit(qsharp.circuit(
    full_circuit._qsharp_factory.program,
    *full_circuit._qsharp_factory.parameter.values(),
    generation_method=CircuitGenerationMethod.Static)
)


In [ ]:
iqpe = create(
    "phase_estimation",
    "qdk_iterative",
)
iqpe.set_algorithm_instance("qpe_circuit_builder", single_qir_circuit_builder)
iqpe.set_algorithm_instance("circuit_executor", ac1000_emulator)

print(f"Running IQPE on Azure Quantum target {target.name}...")
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
)

# Summarize the QPE results
estimated_energy = result.raw_energy
print(f"Energy result: {result.raw_energy}")